In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
apps_cols = pd.read_csv(RAW / "applications.csv", nrows=5).columns
bureau_cols = pd.read_csv(RAW / "credit_bureau.csv", nrows=5).columns
bureau_ids = pd.read_csv(RAW / "credit_bureau.csv", usecols=["id"], dtype={"id": str})["id"]
hold = pd.read_csv(RAW / "holdout_applications.csv", dtype={"id": str}, low_memory=False)

print("holdout shape:", hold.shape)
print("in holdout but not in applications:", sorted(set(hold.columns) - set(apps_cols)))
print("in applications but not in holdout:", sorted(set(apps_cols) - set(hold.columns)))
print("bureau columns present in holdout:", sorted((set(bureau_cols) & set(hold.columns)) - {"id"}))
print("holdout ids found in credit_bureau.csv:", int(hold["id"].isin(bureau_ids).sum()), "of", len(hold))

holdout shape: (5000, 40)
in holdout but not in applications: ['acc_open_past_24mths', 'avg_cur_bal', 'bc_util', 'bureau_recovery_flag', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_high', 'fico_range_low', 'inq_last_6mths', 'mort_acc', 'mths_since_last_delinq', 'num_tl_op_past_12m', 'open_acc', 'pub_rec', 'pub_rec_bankruptcies', 'revol_bal', 'revol_util', 'tot_cur_bal', 'tot_hi_cred_lim', 'total_acc', 'total_rev_hi_lim']
in applications but not in holdout: ['defaulted']
bureau columns present in holdout: ['acc_open_past_24mths', 'avg_cur_bal', 'bc_util', 'bureau_recovery_flag', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_high', 'fico_range_low', 'inq_last_6mths', 'mort_acc', 'mths_since_last_delinq', 'num_tl_op_past_12m', 'open_acc', 'pub_rec', 'pub_rec_bankruptcies', 'revol_bal', 'revol_util', 'tot_cur_bal', 'tot_hi_cred_lim', 'total_acc', 'total_rev_hi_lim']
holdout ids found in credit_bureau.csv: 0 of 5000


In [2]:
import sys
sys.path.append("..")                                   # so `src` can be imported from notebooks/
from src.cleaning import load_training_raw, clean

cleaned = clean(load_training_raw(RAW)).set_index("id")
ref = pd.read_parquet("../data/interim/train_audited.parquet").set_index("id")

print("rows:", len(cleaned), "vs", len(ref))
print("only in module output:", sorted(set(cleaned.columns) - set(ref.columns)))
print("only in notebook output:", sorted(set(ref.columns) - set(cleaned.columns)))

common = [c for c in ref.columns if c in cleaned.columns]
pd.testing.assert_frame_equal(cleaned.loc[ref.index, common], ref[common], check_dtype=False)
print("identical on", len(common), "columns ✅")

flags = ["loan_amnt_corrected", "annual_inc_corrected", "revol_util_corrected", "bc_util_corrected",
         "fico_swapped", "earliest_cr_invalid", "installment_mismatch", "zip_state_mismatch"]
print(cleaned[flags].sum().to_dict())

hold_clean = clean(hold)
print("holdout:", hold_clean.shape, "| leakage column gone:", "bureau_recovery_flag" not in hold_clean.columns)
print("holdout flags:", hold_clean[flags].sum().to_dict())

rows: 30000 vs 30000
only in module output: []
only in notebook output: ['bureau_recovery_flag']
identical on 52 columns ✅
{'loan_amnt_corrected': 300, 'annual_inc_corrected': 899, 'revol_util_corrected': 295, 'bc_util_corrected': 290, 'fico_swapped': 150, 'earliest_cr_invalid': 300, 'installment_mismatch': 11, 'zip_state_mismatch': 27}
holdout: (5000, 52) | leakage column gone: True
holdout flags: {'loan_amnt_corrected': 0, 'annual_inc_corrected': 0, 'revol_util_corrected': 0, 'bc_util_corrected': 0, 'fico_swapped': 0, 'earliest_cr_invalid': 0, 'installment_mismatch': 0, 'zip_state_mismatch': 3}


In [4]:
import importlib, src.features
importlib.reload(src.features)                                    # pick up edits without restarting
from src.features import fit_basic, apply_basic, MODEL_FEATURES, FIT_CUTOFF

cleaned_all = clean(load_training_raw(RAW))
fit_rows = cleaned_all[cleaned_all["issue_dt"] <= FIT_CUTOFF]
params = fit_basic(fit_rows)
feat = apply_basic(cleaned_all, params).set_index("id")

ref = pd.read_parquet("../data/interim/train_a5.parquet").set_index("id")
check_cols = ([f"{c}_cap" for c in params["caps"]] + [f"{c}_log" for c, r in params["caps"].items() if r["log1p"]]
              + ["credit_hist_months", "revol_util", "bc_util", "bc_util_missing", "ever_delinq",
                 "fico_mid", "loan_to_income", "emp_length_years", "purpose_grp", "verification_grp", "purpose_risk"])
pd.testing.assert_frame_equal(feat.loc[ref.index, check_cols], ref[check_cols], check_dtype=False)
print("identical on", len(check_cols), "engineered columns ✅")

spec = json.load(open("../reports/a7_spec.json"))
a3_caps = json.load(open("../reports/a3_caps.json"))
print("caps match A3:", all(abs(params["caps"][c]["upper"] - a3_caps[c]["upper"]) < 1e-9 for c in params["caps"]))
print("purpose list matches A5:", params["purpose_keep"] == spec["purpose_keep"])
print("loan_to_income cap:", round(params["feature_caps"]["loan_to_income"], 6),
      "| spec:", round(spec["feature_caps"]["loan_to_income"], 6))

identical on 26 engineered columns ✅
caps match A3: True
purpose list matches A5: True
loan_to_income cap: 0.466093 | spec: 0.466093


In [5]:
importlib.reload(src.features)
from src.features import fit_basic, apply_basic, fit_latent, apply_latent, FACTOR_SCORE_COLS, PCA_SCORE_COLS

basic_all = apply_basic(cleaned_all, params)
basic_fit = basic_all[basic_all["issue_dt"] <= FIT_CUTOFF]
models = fit_latent(basic_fit)
latent = apply_latent(basic_all, models).set_index("id")

ref6 = pd.read_parquet("../data/interim/train_a6.parquet").set_index("id")
pd.testing.assert_frame_equal(latent.loc[ref6.index, FACTOR_SCORE_COLS], ref6[FACTOR_SCORE_COLS], check_dtype=False)
print("factor scores identical to notebook 06 ✅")

spec6 = json.load(open("../reports/a6_spec.json"))
print("scaler means match A6:", np.allclose(models["scaler"].mean_, spec6["pca"]["mean"]))
print("scaler stds match A6: ", np.allclose(models["scaler"].scale_, spec6["pca"]["std"]))
print("PCA variance shares:", models["pca"].explained_variance_ratio_.round(3))   # expect 0.272 0.142 0.115 0.102
print(latent[PCA_SCORE_COLS + FACTOR_SCORE_COLS].describe().loc[["mean", "std"]].round(3))

/home/kalki/credit-risk-takehome/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/kalki/credit-risk-takehome/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


factor scores identical to notebook 06 ✅
scaler means match A6: True
scaler stds match A6:  True
PCA variance shares: [0.272 0.142 0.115 0.102]
        pc1    pc2   pc3    pc4  factor_debt_size  factor_card_utilization  \
mean  0.018 -0.100  0.03  0.043            -0.004                   -0.062   
std   2.271  1.694  1.52  1.394             0.960                    0.986   

      factor_card_size  factor_credit_seeking  
mean             0.016                  0.049  
std              0.996                  0.947  


In [6]:
import joblib
importlib.reload(src.features)
from src.features import CreditRiskTransform, MODEL_FEATURES, FACTOR_SCORE_COLS, PCA_SCORE_COLS

raw_train = load_training_raw(RAW)
tf = CreditRiskTransform().fit(raw_train)
out = tf.transform(raw_train).set_index("id")

cols = MODEL_FEATURES + PCA_SCORE_COLS + FACTOR_SCORE_COLS
pd.testing.assert_frame_equal(out.loc[latent.index, cols], latent[cols], check_dtype=False)
print("class output identical to the step-by-step functions ✅ | fit rows:", tf.n_fit_rows_)

iv = pd.Series({c: r["iv"] for c, r in tf.woe_bins_.items()}).sort_values(ascending=False).round(3)
print("\nIV with the contract's 5 bins (top 10):"); print(iv.head(10))

h_out = tf.transform(hold)
print("\nholdout transformed:", h_out.shape)
na = h_out[MODEL_FEATURES + PCA_SCORE_COLS + FACTOR_SCORE_COLS].isna().sum()
print("missing values in holdout outputs:", na[na > 0].to_dict())

Path("../contract").mkdir(exist_ok=True)
joblib.dump(tf, "../contract/transform.joblib")
tf2 = joblib.load("../contract/transform.joblib")
pd.testing.assert_frame_equal(tf2.transform(hold), h_out)
print("saved, reloaded, same holdout output ✅ | contract_version", tf2.contract_version)

/home/kalki/credit-risk-takehome/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/kalki/credit-risk-takehome/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/kalki/credit-risk-takehome/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


class output identical to the step-by-step functions ✅ | fit rows: 14373

IV with the contract's 5 bins (top 10):
term_months                0.195
fico_mid                   0.099
loan_to_income             0.098
acc_open_past_24mths       0.075
factor_card_utilization    0.069
factor_credit_seeking      0.067
dti_cap                    0.062
num_tl_op_past_12m         0.049
verification_grp           0.046
factor_debt_size           0.045
dtype: float64

holdout transformed: (5000, 84)
missing values in holdout outputs: {'mths_since_last_delinq': 2378}


/home/kalki/credit-risk-takehome/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/kalki/credit-risk-takehome/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


saved, reloaded, same holdout output ✅ | contract_version 1.0.0


In [7]:
from src.contract import build_data_spec

tf = joblib.load("../contract/transform.joblib")
fit_out = tf.transform(raw_train)
fit_out = fit_out[fit_out["issue_dt"] <= tf.fit_cutoff]

a7 = json.load(open("../reports/a7_spec.json"))
data_spec = build_data_spec(tf, fit_out, a7["exclude"])
with open("../contract/data_spec.json", "w") as f:
    json.dump(data_spec, f, indent=2, default=float)

print("fields:", len(data_spec["fields"]), "| model features:", len(data_spec["model_features"]),
      "| woe bins:", len(data_spec["woe_bins"]))
print(json.dumps(data_spec["fields"][9], indent=2))          # annual_inc, to check the format
print(json.dumps(data_spec["factors"][0], indent=2))

fields: 39 | model features: 29 | woe bins: 33
{
  "name": "annual_inc",
  "type": "float",
  "min": 1000,
  "max": 10000000,
  "role": "model input (as annual_inc_log, loan_to_income)",
  "cleaning": "values below 1,000 treated as thousands (x1,000); capped at p99.5 = 300,000.00 (fitted <= 2014); then log1p",
  "missing_strategy": "required",
  "reliability": "low",
  "reliability_reason": "self-reported; 84% round thousands; verification does not change rounding; implausible extremes ($7.5M 'Verified')"
}
{
  "name": "debt_size",
  "score_column": "factor_debt_size",
  "method": "factor_analysis_promax (minres, 4 factors)",
  "top_loadings": {
    "tot_cur_bal_log": 0.955,
    "tot_hi_cred_lim_log": 0.953,
    "mort_acc": 0.752,
    "annual_inc_log": 0.618
  },
  "variance_explained": 0.184
}


/home/kalki/credit-risk-takehome/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
